# Availability web flow

This notebook follows the **web flow** for recurring training availability using the real Odysseus code.

The aim is deliberately simple:

1. build the real objects the web app needs;
2. see how FastAPI knows about `/availability`;
3. make a real GET request;
4. follow the domain value into the presentation View;
5. make a real POST request;
6. see the redirect back into the GET flow.

The architecture diagrams contain the structural detail. This notebook is the executable companion.

> Select the Odysseus `.venv` as the notebook kernel before running it.


## 1. Imports

We use production classes throughout. There are no mocks or notebook copies of application logic.


In [2]:
from dataclasses import asdict
from pprint import pprint

from fastapi.routing import APIRoute
from fastapi.testclient import TestClient

from odysseus.adapters.in_memory.availability import InMemoryTrainingAvailabilityStore
from odysseus.application.availability import TrainingAvailabilityUseCases
from odysseus.domain.availability import (
    DailyTrainingAvailability,
    RecurringTrainingAvailability,
    Weekday,
)
from odysseus.web.app import create_app
from odysseus.web.presentation.availability import build_training_availability_view

c:\Users\matth\OneDrive\Data Science\portfolio\MC-portfolio\apps\Odysseus\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. Build the real objects

The runtime chain for this example is:

```text
RecurringTrainingAvailability
        |
        v
InMemoryTrainingAvailabilityStore
        |
        v
TrainingAvailabilityUseCases
        |
        v
create_app(...)
        |
        v
FastAPI
```

`create_app(...)` receives the use-case object. The web layer does not construct the store itself.


In [3]:
initial_week = RecurringTrainingAvailability(
    days=(
        DailyTrainingAvailability(
            weekday=Weekday.MONDAY,
            am_available=True,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.TUESDAY,
            am_available=False,
            pm_available=True,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.WEDNESDAY,
            am_available=False,
            pm_available=False,
            is_rest_day=True,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.THURSDAY,
            am_available=False,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.FRIDAY,
            am_available=True,
            pm_available=True,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.SATURDAY,
            am_available=True,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.SUNDAY,
            am_available=False,
            pm_available=False,
            is_rest_day=True,
        ),
    )
)

store = InMemoryTrainingAvailabilityStore(initial_week)
use_cases = TrainingAvailabilityUseCases(store)
app = create_app(use_cases)
client = TestClient(app)

print(type(store).__name__)
print(type(use_cases).__name__)
print(type(app).__name__)

InMemoryTrainingAvailabilityStore
TrainingAvailabilityUseCases
FastAPI


## 3. How FastAPI knows what to call

FastAPI does not infer the meaning of "availability".

`create_app(...)` registers a router. That router registers handler functions against an HTTP **method + path**.

We can inspect the app's registered routes:


In [5]:
for route in app.routes:
    if isinstance(route, APIRoute):
        print(f"{','.join(sorted(route.methods)):8} {route.path}")

If `/availability` appears for both GET and POST, FastAPI has enough information to dispatch browser requests to the correct route handler.

```text
GET  + /availability -> GET handler
POST + /availability -> POST handler
```


## 4. Follow a GET request

The GET flow is:

```text
Browser
-> FastAPI route
-> TrainingAvailabilityUseCases.get_recurring_availability()
-> store.load()
-> RecurringTrainingAvailability
-> presentation builder
-> Jinja template
-> HTML
```


In [6]:
response = client.get("/availability")

print("Status:", response.status_code)
print()
print(response.text[:1200])

Status: 200

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Recurring training availability</title>
  </head>
  <body>
    <main>
      <h1>Recurring training availability</h1>
      <p>Choose AM, PM, or both for training. Choose REST on its own for an explicit rest day. Leave all three unchecked for ordinary unavailability.</p>
      <form method="post" action="http://testserver/availability">
      <table>
        <caption>Current recurring week</caption>
        <thead>
          <tr>
            <th scope="col">Weekday</th>
            <th scope="col">AM</th>
            <th scope="col">PM</th>
            <th scope="col">Rest day</th>
            <th scope="col">Day status</th>
          </tr>
        </thead>
        <tbody>
          
          <tr>
            <th scope="row">Monday</th>
            <td>
              <label>
                <input type="checkbox" name="monday_am" value="on" checked>
                AM available
              </

## 5. See the value before and after presentation

The application returns a **domain value**.

The presentation builder converts that into a **display View** for the web template.


In [7]:
domain_value = use_cases.get_recurring_availability()
view = build_training_availability_view(domain_value)

print("Domain type:", type(domain_value).__name__)
print("View type:", type(view).__name__)

print("\nDomain value:")
pprint(asdict(domain_value))

print("\nDisplay View:")
pprint(asdict(view))

Domain type: RecurringTrainingAvailability
View type: TrainingAvailabilityView

Domain value:
{'days': ({'am_available': True,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.MONDAY: 'monday'>},
          {'am_available': False,
           'is_rest_day': False,
           'pm_available': True,
           'weekday': <Weekday.TUESDAY: 'tuesday'>},
          {'am_available': False,
           'is_rest_day': True,
           'pm_available': False,
           'weekday': <Weekday.WEDNESDAY: 'wednesday'>},
          {'am_available': False,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.THURSDAY: 'thursday'>},
          {'am_available': True,
           'is_rest_day': False,
           'pm_available': True,
           'weekday': <Weekday.FRIDAY: 'friday'>},
          {'am_available': True,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.SATURDAY

The distinction is:

```text
RecurringTrainingAvailability = business truth
TrainingAvailabilityView      = display-friendly shape
```

The View does not replace or re-validate the domain model.


## 6. Inspect the form fields rendered by the real template

Rather than guessing which HTML names exist, inspect the actual page.


In [8]:
import re

field_names = sorted(set(re.findall(r'name=["\']([^"\']+)["\']', response.text)))

for field_name in field_names:
    print(field_name)

friday_am
friday_pm
friday_rest
monday_am
monday_pm
monday_rest
saturday_am
saturday_pm
saturday_rest
sunday_am
sunday_pm
sunday_rest
thursday_am
thursday_pm
thursday_rest
tuesday_am
tuesday_pm
tuesday_rest
wednesday_am
wednesday_pm
wednesday_rest


Those are the browser field names that the POST route receives.

`forms.py` exists to translate that HTTP representation into the real domain representation.


## 7. Follow a POST request

Submitting no availability checkboxes gives a useful simple example: every day becomes ordinarily unavailable rather than an explicit rest day.

The important part here is the web flow, not the particular training choice.


In [9]:
post_response = client.post(
    "/availability",
    data={},
    follow_redirects=False,
)

print("POST status:", post_response.status_code)
print("Redirect location:", post_response.headers.get("location"))

POST status: 415
Redirect location: None


A successful POST redirects instead of rendering the final page directly.

That produces:

```text
POST /availability
-> parse form values
-> construct domain value
-> update use case
-> store.save(...)
-> redirect to GET /availability
```


## 8. Follow the redirect with a GET

Now ask for the page again and inspect the domain value stored by the application.


In [10]:
after_update = client.get("/availability")
stored_week = use_cases.get_recurring_availability()

print("GET status:", after_update.status_code)
print()
pprint(asdict(stored_week))

GET status: 200

{'days': ({'am_available': True,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.MONDAY: 'monday'>},
          {'am_available': False,
           'is_rest_day': False,
           'pm_available': True,
           'weekday': <Weekday.TUESDAY: 'tuesday'>},
          {'am_available': False,
           'is_rest_day': True,
           'pm_available': False,
           'weekday': <Weekday.WEDNESDAY: 'wednesday'>},
          {'am_available': False,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.THURSDAY: 'thursday'>},
          {'am_available': True,
           'is_rest_day': False,
           'pm_available': True,
           'weekday': <Weekday.FRIDAY: 'friday'>},
          {'am_available': True,
           'is_rest_day': False,
           'pm_available': False,
           'weekday': <Weekday.SATURDAY: 'saturday'>},
          {'am_available': False,
           'is_rest_day': T

Because the POST updated the same injected store used by the GET flow, the subsequent GET sees the new value.

That is the complete current web loop:

```text
GET:
Browser
-> Route
-> Use case
-> Store
-> Domain value
-> Presentation
-> Template
-> Browser

POST:
Browser form
-> Route
-> Forms translation
-> Domain value
-> Use case
-> Store
-> Redirect
-> GET flow
```

The important responsibility split is:

- **route** = coordinate HTTP flow;
- **forms** = translate incoming browser values;
- **domain** = decide what is valid training availability;
- **use case** = coordinate the application operation;
- **presentation** = prepare outgoing display data;
- **template** = render HTML.
